# LibreOffice-Format-Konverter (mit Gradio-Oberfläche)

Konvertiert LibreOffice-Formate über **Microsoft Office** in moderne Formate:
- `.odt` → `.docx` (via Word)
- `.ods` → `.xlsx` (via Excel)
- `.odp` → `.pptx` (via PowerPoint)

**Voraussetzung:** Microsoft Office + `pywin32`

Die **Gradio-Oberfläche** zeigt live den Fortschritt und erlaubt den CSV-Export der Ergebnisse.

In [ ]:
# ── Abhängigkeiten prüfen / installieren ──

try:
    import win32com.client
    print("✅ pywin32")
except ImportError:
    %pip install pywin32 -q
    import win32com.client
    print("✅ pywin32 installiert")

try:
    import gradio as gr
    print(f"✅ gradio {gr.__version__}")
except ImportError:
    %pip install gradio -q
    import gradio as gr
    print(f"✅ gradio {gr.__version__} installiert")

In [ ]:
import os
import csv
import shutil
import tempfile
from pathlib import Path
from datetime import datetime
import pythoncom

IGNORIERTE_VERZEICHNISSE = {
    "node_modules", ".git", "__pycache__", ".ipynb_checkpoints",
    ".venv", "venv", "env", ".env", "_lo_backup", "_office_backup",
}


# ════════════════════════════════════════════════════════════════
# 1. Dateien finden
# ════════════════════════════════════════════════════════════════

def lo_dateien_finden(basis_pfad: str) -> dict[str, list[Path]]:
    """Findet rekursiv alle .odt, .ods, .odp Dateien."""
    basis = Path(basis_pfad).resolve()
    gefunden = {"odt": [], "ods": [], "odp": []}
    endungen = {".odt": "odt", ".ods": "ods", ".odp": "odp"}

    for verz, unterverz, dateien in os.walk(basis):
        unterverz[:] = [
            d for d in unterverz
            if d not in IGNORIERTE_VERZEICHNISSE and not d.startswith(".")
        ]
        for name in sorted(dateien):
            if name.startswith("~") or name.startswith(".~"):
                continue
            nl = name.lower()
            for endung, typ in endungen.items():
                if nl.endswith(endung):
                    gefunden[typ].append(Path(verz) / name)

    return gefunden


# ════════════════════════════════════════════════════════════════
# 2. Backup-Helfer
# ════════════════════════════════════════════════════════════════

def _backup_verschieben(quelle: Path, backup_ordner: Path):
    if not backup_ordner:
        return
    backup_ordner.mkdir(parents=True, exist_ok=True)
    ziel = backup_ordner / quelle.name
    if ziel.exists():
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        ziel = backup_ordner / f"{quelle.stem}_{ts}{quelle.suffix}"
    shutil.move(str(quelle), str(ziel))


# ════════════════════════════════════════════════════════════════
# 3. Konvertier-Funktionen (als Generatoren für Live-Feedback)
# ════════════════════════════════════════════════════════════════

def konvertiere_word(dateien: list[Path], backup_ordner: Path = None):
    """Generator: konvertiert .odt → .docx, yielded (log_zeile, ergebnis_dict)."""
    if not dateien:
        return
    pythoncom.CoInitialize()
    word = None
    try:
        word = win32com.client.Dispatch("Word.Application")
        word.Visible = False
        word.DisplayAlerts = False
        for i, quelle in enumerate(dateien, 1):
            ziel = quelle.with_suffix(".docx")
            prefix = f"[Word {i}/{len(dateien)}]"
            if ziel.exists():
                yield f"{prefix} ⏭️ {quelle.name}  →  .docx existiert bereits", \
                      {"quelle": str(quelle), "ziel": str(ziel), "status": "übersprungen", "typ": "odt"}
                continue
            try:
                yield f"{prefix} ⏳ {quelle.name}  →  konvertiere …", None
                doc = word.Documents.Open(str(quelle))
                doc.SaveAs2(str(ziel), FileFormat=16)
                doc.Close()
                yield f"{prefix} ✅ {quelle.name}  →  {ziel.name}", \
                      {"quelle": str(quelle), "ziel": str(ziel), "status": "ok", "typ": "odt"}
                _backup_verschieben(quelle, backup_ordner)
            except Exception as ex:
                yield f"{prefix} ❌ {quelle.name}  →  {ex}", \
                      {"quelle": str(quelle), "status": "fehler", "grund": str(ex)[:120], "typ": "odt"}
    finally:
        if word:
            try: word.Quit()
            except: pass
        pythoncom.CoUninitialize()


def konvertiere_powerpoint(dateien: list[Path], backup_ordner: Path = None):
    """Generator: konvertiert .odp → .pptx."""
    if not dateien:
        return
    pythoncom.CoInitialize()
    ppt_app = None
    try:
        ppt_app = win32com.client.Dispatch("PowerPoint.Application")
        for i, quelle in enumerate(dateien, 1):
            ziel = quelle.with_suffix(".pptx")
            prefix = f"[PowerPoint {i}/{len(dateien)}]"
            if ziel.exists():
                yield f"{prefix} ⏭️ {quelle.name}  →  .pptx existiert bereits", \
                      {"quelle": str(quelle), "ziel": str(ziel), "status": "übersprungen", "typ": "odp"}
                continue
            try:
                yield f"{prefix} ⏳ {quelle.name}  →  konvertiere …", None
                prs = ppt_app.Presentations.Open(str(quelle), WithWindow=False)
                prs.SaveAs(str(ziel), FileFormat=24)
                prs.Close()
                yield f"{prefix} ✅ {quelle.name}  →  {ziel.name}", \
                      {"quelle": str(quelle), "ziel": str(ziel), "status": "ok", "typ": "odp"}
                _backup_verschieben(quelle, backup_ordner)
            except Exception as ex:
                yield f"{prefix} ❌ {quelle.name}  →  {ex}", \
                      {"quelle": str(quelle), "status": "fehler", "grund": str(ex)[:120], "typ": "odp"}
    finally:
        if ppt_app:
            try: ppt_app.Quit()
            except: pass
        pythoncom.CoUninitialize()


def konvertiere_excel(dateien: list[Path], backup_ordner: Path = None):
    """Generator: konvertiert .ods → .xlsx."""
    if not dateien:
        return
    pythoncom.CoInitialize()
    excel = None
    try:
        excel = win32com.client.Dispatch("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        for i, quelle in enumerate(dateien, 1):
            ziel = quelle.with_suffix(".xlsx")
            prefix = f"[Excel {i}/{len(dateien)}]"
            if ziel.exists():
                yield f"{prefix} ⏭️ {quelle.name}  →  .xlsx existiert bereits", \
                      {"quelle": str(quelle), "ziel": str(ziel), "status": "übersprungen", "typ": "ods"}
                continue
            try:
                yield f"{prefix} ⏳ {quelle.name}  →  konvertiere …", None
                wb = excel.Workbooks.Open(str(quelle))
                wb.SaveAs(str(ziel), FileFormat=51)
                wb.Close()
                yield f"{prefix} ✅ {quelle.name}  →  {ziel.name}", \
                      {"quelle": str(quelle), "ziel": str(ziel), "status": "ok", "typ": "ods"}
                _backup_verschieben(quelle, backup_ordner)
            except Exception as ex:
                yield f"{prefix} ❌ {quelle.name}  →  {ex}", \
                      {"quelle": str(quelle), "status": "fehler", "grund": str(ex)[:120], "typ": "ods"}
    finally:
        if excel:
            try: excel.Quit()
            except: pass
        pythoncom.CoUninitialize()


# ════════════════════════════════════════════════════════════════
# 4. CSV erzeugen
# ════════════════════════════════════════════════════════════════

def csv_erzeugen(ergebnisse: list[dict], ordner_pfad: str) -> str:
    """Schreibt CSV und gibt den Dateipfad zurück."""
    csv_zeilen = [
        e for e in ergebnisse
        if e["status"] in ("ok", "übersprungen") and "ziel" in e
    ]
    csv_pfad = str(Path(ordner_pfad).resolve() / "konvertierte_lo_dateien.csv")

    with open(csv_pfad, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f, delimiter=";")
        writer.writerow(["dateiname_neu", "pfad", "typ", "quelle_original", "status"])
        for e in csv_zeilen:
            ziel = Path(e["ziel"])
            writer.writerow([
                ziel.name,
                str(ziel.parent),
                e["typ"],
                Path(e["quelle"]).name,
                e["status"],
            ])

    return csv_pfad


print("✅ Alle Funktionen geladen.")

## Gradio-Oberfläche starten

Die nächste Zelle öffnet die Oberfläche. Der Ablauf:

1. **Pfad eingeben** und auf *Dateien suchen* klicken → Vorschau der gefundenen Dateien
2. **Konvertierung starten** → Live-Protokoll zeigt jede einzelne Datei
3. **CSV herunterladen** über den Download-Button

In [ ]:
# ════════════════════════════════════════════════════════════════
# Gradio-App
# ════════════════════════════════════════════════════════════════

# Globaler Zustand: gefundene Dateien zwischen Suche und Konvertierung teilen
_gefunden = {}


def dateien_suchen(ordner_pfad: str):
    """Schritt 1: Ordner durchsuchen und Vorschau liefern."""
    global _gefunden
    ordner_pfad = ordner_pfad.strip()

    if not ordner_pfad:
        return "⚠️ Bitte einen Ordnerpfad eingeben.", [], gr.update(interactive=False)

    basis = Path(ordner_pfad)
    if not basis.exists():
        return f"❌ Ordner existiert nicht:\n{basis}", [], gr.update(interactive=False)
    if not basis.is_dir():
        return f"❌ Kein Ordner:\n{basis}", [], gr.update(interactive=False)

    _gefunden = lo_dateien_finden(ordner_pfad)
    gesamt = sum(len(v) for v in _gefunden.values())

    if gesamt == 0:
        return "✅ Keine .odt / .ods / .odp Dateien gefunden.", [], gr.update(interactive=False)

    # Vorschau-Tabelle aufbauen
    typ_label = {"odt": ".odt → .docx", "ods": ".ods → .xlsx", "odp": ".odp → .pptx"}
    tabelle = []
    for typ in ["odt", "ods", "odp"]:
        for d in _gefunden[typ]:
            kb = d.stat().st_size / 1024
            tabelle.append([typ_label[typ], d.name, str(d.parent), f"{kb:.0f} KB"])

    zusammenfassung = (
        f"📂 {gesamt} Dateien gefunden:\n"
        f"    .odt: {len(_gefunden['odt'])}  │  "
        f".ods: {len(_gefunden['ods'])}  │  "
        f".odp: {len(_gefunden['odp'])}"
    )
    return zusammenfassung, tabelle, gr.update(interactive=True)


def konvertierung_starten(ordner_pfad: str, backup: bool):
    """Schritt 2: Generator – yielded (log, tabelle, csv_pfad) bei jedem Schritt."""
    global _gefunden
    ordner_pfad = ordner_pfad.strip()

    if not _gefunden or sum(len(v) for v in _gefunden.values()) == 0:
        yield "⚠️ Erst Dateien suchen!", [], None
        return

    basis = Path(ordner_pfad).resolve()
    backup_ordner = (basis / "_lo_backup") if backup else None

    log_zeilen = []
    ergebnisse = []
    tabelle = []

    if backup_ordner:
        log_zeilen.append(f"📦 Backup-Ordner: {backup_ordner}")
    log_zeilen.append(f"")
    log_zeilen.append(f"{'═'*55}")
    log_zeilen.append(f"  Konvertierung gestartet – {datetime.now():%H:%M:%S}")
    log_zeilen.append(f"{'═'*55}")
    log_zeilen.append("")

    # Generatoren durchlaufen
    generatoren = [
        ("odt", konvertiere_word(_gefunden["odt"], backup_ordner)),
        ("odp", konvertiere_powerpoint(_gefunden["odp"], backup_ordner)),
        ("ods", konvertiere_excel(_gefunden["ods"], backup_ordner)),
    ]

    for typ, gen in generatoren:
        for log_msg, erg in gen:
            log_zeilen.append(log_msg)
            if erg:
                ergebnisse.append(erg)
                status_icon = {"ok": "✅", "übersprungen": "⏭️", "fehler": "❌"}
                tabelle.append([
                    status_icon.get(erg["status"], "?"),
                    Path(erg["quelle"]).name,
                    Path(erg.get("ziel", "")).name if erg.get("ziel") else "—",
                    erg["status"],
                ])
            # Live-Update an Gradio
            yield "\n".join(log_zeilen), tabelle, None

    # Zusammenfassung
    ok     = sum(1 for e in ergebnisse if e["status"] == "ok")
    fehler = sum(1 for e in ergebnisse if e["status"] == "fehler")
    skip   = sum(1 for e in ergebnisse if e["status"] == "übersprungen")

    log_zeilen.append("")
    log_zeilen.append(f"{'═'*55}")
    log_zeilen.append(f"  Fertig – {datetime.now():%H:%M:%S}")
    log_zeilen.append(f"{'═'*55}")
    log_zeilen.append(f"  ✅ Konvertiert:  {ok}")
    log_zeilen.append(f"  ⏭️ Übersprungen: {skip}")
    log_zeilen.append(f"  ❌ Fehler:       {fehler}")
    log_zeilen.append(f"{'═'*55}")

    # CSV erzeugen
    csv_pfad = None
    if ok > 0 or skip > 0:
        csv_pfad = csv_erzeugen(ergebnisse, ordner_pfad)
        log_zeilen.append(f"")
        log_zeilen.append(f"📋 CSV gespeichert: {csv_pfad}")

    yield "\n".join(log_zeilen), tabelle, csv_pfad


# ── Gradio UI ──────────────────────────────────────────────

with gr.Blocks(
    title="LibreOffice-Konverter",
    theme=gr.themes.Soft(),
) as app:

    gr.Markdown("## 📄 LibreOffice → Office-Konverter")
    gr.Markdown(
        ".odt → .docx &nbsp;│&nbsp; .ods → .xlsx &nbsp;│&nbsp; .odp → .pptx "
        "&nbsp;&nbsp;*(via Microsoft Office)*"
    )

    with gr.Row():
        pfad_input = gr.Textbox(
            label="Ordnerpfad",
            placeholder=r"z.B. D:\03_LehramtT",
            scale=4,
        )
        backup_check = gr.Checkbox(
            label="Backup erstellen",
            value=True,
            scale=1,
        )

    with gr.Row():
        btn_suchen = gr.Button("🔍 Dateien suchen", variant="secondary")
        btn_start  = gr.Button("🚀 Konvertierung starten", variant="primary", interactive=False)

    such_status = gr.Textbox(label="Suchergebnis", interactive=False, lines=3)

    vorschau_tabelle = gr.Dataframe(
        headers=["Konvertierung", "Dateiname", "Ordner", "Größe"],
        label="Gefundene Dateien",
        interactive=False,
        wrap=True,
    )

    gr.Markdown("---")
    gr.Markdown("### Konvertierung")

    log_output = gr.Textbox(
        label="Live-Protokoll",
        interactive=False,
        lines=18,
        max_lines=40,
        autoscroll=True,
    )

    ergebnis_tabelle = gr.Dataframe(
        headers=["Status", "Quelldatei", "Neue Datei", "Ergebnis"],
        label="Ergebnisse",
        interactive=False,
        wrap=True,
    )

    csv_download = gr.File(label="📥 CSV herunterladen", visible=True)

    # Events
    btn_suchen.click(
        fn=dateien_suchen,
        inputs=[pfad_input],
        outputs=[such_status, vorschau_tabelle, btn_start],
    )

    btn_start.click(
        fn=konvertierung_starten,
        inputs=[pfad_input, backup_check],
        outputs=[log_output, ergebnis_tabelle, csv_download],
    )


app.launch()